In [ ]:
from pathlib import Path
import pydicom
import numpy as np
import re
import os
import pandas as pd

In [ ]:
cwd = Path.cwd()
print("Current working directory:", cwd) # /path/to/home

root_path = Path("/path/to/BrainWear_Kareem")/"BrainWear MRI Scans"/"<scan_id>"
print("Root path for analysis:", root_path)
print("Path exists:", root_path.exists()) 
print("Is a directory:", root_path.is_dir())

In [ ]:
def to_list(val):
    """Helper to standardize single values vs MultiValues into a list."""
    if isinstance(val, str):
        return [val]
    elif isinstance(val, pydicom.multival.MultiValue):
        return list(val)
    return val

def check_T2_SE(ds):
    desc = ds.get("SeriesDescription", "").upper()
    scan_seq = to_list(ds.get("ScanningSequence", []))

    # Get Image Type (0008,0008)
    # Scan 1 Example: ['DERIVED', 'PRIMARY', 'DIFFUSION', 'ADC', ...]
    img_types = [x.upper() for x in to_list(ds.get("ImageType", []))]
    
    # Reject diffusion scans
    if "DIFFUSION" in img_types: return False
    if "ADC" in img_types: return False
    if "DERIVED" in img_types: return False
        
    if "GR" in scan_seq:
        return False  # Reject Gradient Echo

    # Reject if Inversion Recovery (FLAIR)
    if "IR" in scan_seq and ("TIRM" in desc or "FLAIR" in desc or "DARK" in desc):
        return False
    
    # Exclude 3D Volumes (VISTA, CUBE, SPACE)
    acq_type = ds.get("MRAcquisitionType", "2D") # Default to 2D if missing, but usually present
    if acq_type == "3D":
        return False
        
    # Secondary check for 3D keywords in description if tag is missing
    if "VISTA" in desc or "CUBE" in desc or "SPACE" in desc or "3D" in desc:
        return False

    # Check Echo Time (0018,0081)
    # T2 weighted images have Long TE (usually > 80ms)
    # T2* or Proton Density will have shorter TE
    te = ds.get("EchoTime", 0)
    if te < 80: 
        return False

    # Check Repitition Time (0018,0080)
    # Standard T2 has very long TR (> 2000ms)
    tr = ds.get("RepetitionTime", 0)
    if tr < 2000:
        return False

    return True

In [ ]:
def get_scan_plane(orientation_vectors):
    """
    Determines the MRI scan plane from Image Orientation (Patient) tag.
    Input: List or array of 6 floats [rx, ry, rz, cx, cy, cz]
    """
    # Split the 6 values into Row (v1) and Column (v2) vectors
    v1 = np.array(orientation_vectors[:3])
    v2 = np.array(orientation_vectors[3:])
    
    # Calculate the Normal Vector (cross product)
    normal = np.cross(v1, v2)
    
    # Take the absolute values to find the dominant direction
    abs_normal = np.absolute(normal)
    max_index = np.argmax(abs_normal)
    
    # Map the dominant axis to the plane name
    # DICOM LPS System: 0=X (Left), 1=Y (Posterior), 2=Z (Superior)
    if max_index == 0:
        return "Sagittal"
    elif max_index == 1:
        return "Coronal"
    elif max_index == 2:
        return "Axial"
    else:
        return "Unknown"

def get_scan_details(file_path):
    ds = pydicom.dcmread(file_path, stop_before_pixels=True)
    # Get Weighting from Description
    desc = ds.get("SeriesDescription", "").upper()
    image_type = ds.get("ImageType", "")
    weighting = "Other"
    if "T1" in desc: weighting = "T1"
    elif "FLAIR" in desc: weighting = "FLAIR"
    elif "T2" in desc:
        t2_se = check_T2_SE(ds)
        weighting = "T2" if t2_se else "Other T2"
    elif "DIFF" in desc or "DIFFUSION" in image_type: weighting = "DWI"
    
    # Calculate Plane from Orientation Tag (0020, 0037)
    orientation = ds.get("ImageOrientationPatient", None)
    plane = "Unknown"
    
    if orientation:
        plane = get_scan_plane(orientation)

    return weighting, plane

In [ ]:
def extract_dicom_metadata(file_path, full_results=False):
    try:
        print(f"{'='*60}")
        print(f" DICOM METADATA SUMMARY")
        print(f"{'='*60}")

        if not full_results:
            weight, plane = get_scan_details(file_path)
            print(f"Calculated Modality: {weight}")
            print(f"Calculated Plane:    {plane}")

        else:
            ds = pydicom.dcmread(file_path, stop_before_pixels=True)

            print(f"Patient Name:    {ds.get('PatientName', 'N/A')}")
            print(f"Patient ID:      {ds.get('PatientID', 'N/A')}")
            print(f"Modality:        {ds.get('Modality', 'N/A')}")
            print(f"Study Date:      {ds.get('StudyDate', 'N/A')}")
            print(f"Image Size:      {ds.Rows} x {ds.Columns} pixels")

            print(f"\n{'='*60}")
            print(f" ALL ENCAPSULATED TAGS")
            print(f"{'='*60}")

            for element in ds:
                if element.tag == (0x7fe0, 0x0010): # Skip pixel data
                    print(f"{element.tag} [Pixel Data] : <Binary Data Buffered>")
                    continue
                
                print(f"{element.tag} {element.name:35}: {element.value}")

    except FileNotFoundError:
        print("Error: The file path provided does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
def compare_metadata(path_to_scan_1, path_to_scan_2, full_metadata=False):
    try:
        ds_1 = pydicom.dcmread(path_to_scan_1, stop_before_pixels=True)
        ds_2 = pydicom.dcmread(path_to_scan_2, stop_before_pixels=True)

        print(f"========== SUMMARY ==============")
        print(f"Patient Name:   {ds_1.get('PatientName', 'N/A')} vs {ds_2.get('PatientName', 'N/A')}")
        print(f"Patient ID:     {ds_1.get('PatientID', 'N/A')} vs {ds_2.get('PatientID', 'N/A')}")
        print(f"Modality:       {ds_1.get('Modality', 'N/A')} vs {ds_2.get('Modality', 'N/A')}")
        print(f"Study Date:     {ds_1.get('StudyDate', 'N/A')} vs {ds_2.get('StudyDate', 'N/A')}")
        print(f"Image Size:     {ds_1.Rows} x {ds_1.Columns} vs {ds_2.Rows} x {ds_2.Columns} pixels")

        weight_1, plane_1 = get_scan_details(path_to_scan_1)
        weight_2, plane_2 = get_scan_details(path_to_scan_2)
        print(f"Calculated Modality: {weight_1} vs {weight_2}")
        print(f"Calculated Plane:    {plane_1} vs {plane_2}")

        if full_metadata:
            print(f"\n{'='*60}")
            print(f" COMPARING METADATA BETWEEN SCANS")
            print(f"{'='*60}")

            for element in ds_1:
                if element.tag == (0x7fe0, 0x0010): # Skip pixel data
                    continue
                value_1 = element.value
                # Get just the value, not the entire DataElement
                if element.tag in ds_2:
                    value_2 = ds_2[element.tag].value
                else:
                    value_2 = "Not Present"
                if value_1 != value_2:
                    print(f"{element.tag} {element.name:35}: {value_1} vs {value_2}")

    except FileNotFoundError:
        print("Error: The file path provided does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")


In [ ]:
def is_date_dir(path):
    """Checks if a directory name matches YYYYMMDD format."""
    return path.is_dir() and re.match(r"\d{8}$", path.name)

def get_earliest_date_subdir(patient_path):
    """Finds the earliest YYYYMMDD subdirectory within a patient folder."""
    date_dirs = [p for p in patient_path.iterdir() if is_date_dir(p)]
    
    if not date_dirs:
        return None
    
    date_dirs.sort(key=lambda x: x.name)
    return date_dirs[0]

def get_all_date_subdirs(patient_path):
    """Returns a list of all YYYYMMDD subdirectories."""
    # Find all dirs matching 8 digits
    return [p for p in patient_path.iterdir() if p.is_dir() and re.match(r"\d{8}$", p.name)]

def find_patient_dirs_fast(root_path):
    # Iterate through directory tree
    for dirpath, dirnames, filenames in os.walk(root_path):
        # iterate over copy of dirnames to allow modification
        for dirname in dirnames[:]:
            if re.match(r"BW-\d+", dirname):
                yield Path(dirpath) / dirname
                # Optimization: Don't search inside the patient folder
                dirnames.remove(dirname)

In [ ]:
def paths_to_axial_t2_scans(root_path, debug=False):
    axial_t2_scans = {}
    no_axial_t2 = []
    gr_1_axial_t2 = []

    for patient_dir in find_patient_dirs_fast(root_path):
        patient_id = patient_dir.name  # e.g. "BW-001"

        earliest_date_dir = get_earliest_date_subdir(patient_dir)
        if earliest_date_dir is None:
            if debug:
                print(f"[SKIP] {patient_id}: No date subdirectory found")
            continue

        mr_dir = earliest_date_dir / "MR"
        if not mr_dir.is_dir():
            if debug:
                print(f"[SKIP] {patient_id}: No MR directory in {earliest_date_dir.name}")
            continue

        valid_paths = []
        for scan_dir in sorted(mr_dir.iterdir()):
            if not scan_dir.is_dir():
                continue

            # Grab the first .dcm file to determine weighting and plane
            dcm_files = list(scan_dir.glob("*.dcm"))
            if not dcm_files:
                continue

            try:
                weighting, plane = get_scan_details(dcm_files[0])
            except Exception as e:
                if debug:
                    print(f"[WARN] {patient_id}/{scan_dir.name}: Could not read DICOM — {e}")
                continue

            if weighting == "T2" and plane == "Axial":
                valid_paths.append(scan_dir)

        if valid_paths:
            axial_t2_scans[patient_id] = valid_paths
            if debug:
                print(f"[OK]   {patient_id}: {len(valid_paths)} axial T2 scan(s) found")
            if len(valid_paths) > 1:
                gr_1_axial_t2.append(patient_id)
                print(valid_paths)
        else:
            no_axial_t2.append(patient_id)
            if debug:
                print(f"[SKIP] {patient_id}: No axial T2 scans")

    print(f"Total patients with axial T2-SE scans: {len(axial_t2_scans)}")
    print(f"Patients without axial T2-SE scans: {no_axial_t2}")
    print(f"Patients with more than one axial T2-SE scan: {gr_1_axial_t2}")

    return list(axial_t2_scans.keys()), axial_t2_scans


In [ ]:
import dicom2nifti
import json

def convert_and_extract_metadata(patient_id, dicom_folder, output_path):
    # Convert DICOM to NIfTI
    print(f"Converting DICOMs for patient {patient_id}...")
    dicom2nifti.convert_directory(dicom_folder, output_path, compression=False, reorient=True)
    # output_filename = output_path/"T2_axial.nii"
    # dicom2nifti.convert_dicom.dicom_series_to_nifti(
    #     reorient_nifti=True,
    #     output_file=output_filename,
    #     original_dicom_directory=dicom_folder
    # )
    
    # Extract Metadata manually
    # Read the first DICOM file in the folder to get the tags
    first_file = next((f for f in os.listdir(dicom_folder) if f.endswith('.dcm')), None)
    
    if first_file:
        ds = pydicom.dcmread(os.path.join(dicom_folder, first_file))
        
        # Create a dictionary of the metadata you want to keep
        # Note: You cannot blindly dump 'ds' to JSON because some DICOM data (like pixel arrays) is not serializable.
        metadata = {
            "PatientID": str(ds.get("PatientID", "Unknown")),
            "SeriesDescription": str(ds.get("SeriesDescription", "Unknown")),
            "Manufacturer": str(ds.get("Manufacturer", "Unknown")),
            "ManufacturersModelName": str(ds.get("ManufacturerModelName", "Unknown")),
            "SoftwareVersions": str(ds.get("SoftwareVersions", "Unknown")),
            "MagneticFieldStrength": float(ds.get("MagneticFieldStrength", 0)),
            "ReceiveCoilName": str(ds.get("ReceiveCoilName", "Unknown")),
            
            # Physics
            "RepetitionTime": float(ds.get("RepetitionTime", 0)),
            "EchoTime": float(ds.get("EchoTime", 0)),
            "FlipAngle": float(ds.get("FlipAngle", 0)),
            
            # Geometry
            "SliceThickness": float(ds.get("SliceThickness", 0)),
            "SpacingBetweenSlices": float(ds.get("SpacingBetweenSlices", 0)),
            "AcquisitionMatrix": ds.get("AcquisitionMatrix", ""),
            
            # Subject
            "PatientSex": str(ds.get("PatientSex", "Unknown")),
            "PatientAge": str(ds.get("PatientAge", "Unknown")),
            "PatientWeight": float(ds.get("PatientWeight", 0)),
        }
        
        # Save metadata to JSON
        json_path = os.path.join(output_path, "metadata.json")
        with open(json_path, 'w') as f:
            json.dump(metadata, f, indent=4)
        # print(f"Metadata saved to {json_path}")
    else:
        print("No .dcm files found to extract metadata.")

In [ ]:
root_path = Path("/path/to/BrainWear_Kareem")/"BrainWear MRI Scans"/"<scan_id>"
print(f"Root path for analysis: {root_path}")
print(f"Path exists {root_path.exists()}") 
print(f"Is a directory: {root_path.is_dir()}")
output_parent_dir = Path("/path/to/BrainWear_Kareem")/"Processed_Brainwear"
print(f"Output path: {output_parent_dir}")
print("================================\n")

patient_ids, paths = paths_to_axial_t2_scans(root_path, debug=False)
print("================================\n")

In [ ]:
for patient_id in patient_ids:
    output_dir_name = patient_id.replace("-", "_")
    output_dir = output_parent_dir/output_dir_name
    os.makedirs(output_dir, exist_ok=True)
    convert_and_extract_metadata(patient_id, paths[patient_id][0], output_dir)

# Check expected number of processed files
output_subfolders = [f for f in Path(output_parent_dir).iterdir() if f.is_dir()]
assert len(output_subfolders) == len(patient_ids), f"Expected {len(patient_ids)} patient folders in the processed Brainwear folder, found {len(output_subfolders)}."
print(f"Found {len(output_subfolders)} patient folders. All present")

In [ ]:
def rename_files(root_folder, base_name="scan", extension=".nii"):
    """
    Rename all files of a certain extension in subfolders of root_folder.
    
    Args:
        root_folder (str): The root folder to search through
        base_name (str): The base name for renamed files (default: "scan")
                        If multiple files with the same extension exist in a subfolder, they'll be numbered
                        as: base_name_001.nii, base_name_002.nii, etc.
    
    Returns:
        dict: A dictionary with subfolder paths as keys and lists of (old_name, new_name) tuples as values
    """
    root_path = Path(root_folder)
    rename_log = {}
    
    if not root_path.exists():
        raise ValueError(f"Folder does not exist: {root_folder}")
    
    # Iterate through all subfolders
    for subfolder in root_path.rglob("*"):
        if subfolder.is_dir():
            # Find all files with the given extension in this subfolder
            dcm_files = sorted(subfolder.glob(f"*{extension}"))
            
            if len(dcm_files) == 0:
                continue
            
            rename_log[str(subfolder)] = []
            
            # Rename files
            if len(dcm_files) == 1:
                # Single file - rename without numbering
                old_path = dcm_files[0]
                new_name = f"{base_name}{extension}"
                new_path = subfolder / new_name
                
                # Only rename if the name is different
                if old_path.name != new_name:
                    old_path.rename(new_path)
                    rename_log[str(subfolder)].append((old_path.name, new_name))
                    # print(f"Renamed: {old_path} -> {new_path}")
            else:
                # Multiple files - add sequential numbering
                for idx, old_path in enumerate(dcm_files, start=1):
                    new_name = f"{base_name}_{idx:03d}{extension}"
                    new_path = subfolder / new_name
                    
                    # Only rename if the name is different
                    if old_path.name != new_name:
                        old_path.rename(new_path)
                        rename_log[str(subfolder)].append((old_path.name, new_name))
                        # print(f"Renamed: {old_path} -> {new_path}")
    print(f"Renaming complete. Total subfolders processed: {len(rename_log)}")
    # return rename_log


In [ ]:
rename_files(output_parent_dir, base_name="Axial_T2", extension=".nii")